# Prepare Data for LLM Finetuning

This notebook reads the collected JSON data (`collected_data.json` and `best_config.json`) for PostgreSQL and MySQL from `/home/E2ETune-AI4DB/data/postgresql` and `/home/E2ETune-AI4DB/data/mysql`.

It groups the data by database and generates two output files `postgres_combined_data.json` and `mysql_combined_data.json` incorporating new variables such as `hardware_specs` along with the base benchmark details.

In [1]:
import os
import json

def process_database(db_path, db_name):
    """
    Reads all available collected_data.json and best_config.json
    under the database path to assemble the final representation.
    """
    combined_data = []
    
    if not os.path.exists(db_path):
        print(f"Path does not exist: {db_path}")
        return combined_data
        
    # Iterate through hardware spec directories (e.g., hetzner-4c-8t-32gb)
    hardware_specs = [d for d in os.listdir(db_path) if os.path.isdir(os.path.join(db_path, d))]
    
    for hw in hardware_specs:
        hw_path = os.path.join(db_path, hw)
        # Iterate over benchmarks (e.g., job, tpcc, ssb)
        benchmarks = [d for d in os.listdir(hw_path) if os.path.isdir(os.path.join(hw_path, d))]
        
        for bench in benchmarks:
            bench_path = os.path.join(hw_path, bench)
            # Iterate over workload directories (e.g., job_0, tpcc_10)
            workloads = [d for d in os.listdir(bench_path) if os.path.isdir(os.path.join(bench_path, d))]
            
            for workload in workloads:
                workload_dir = os.path.join(bench_path, workload)
                collected_data_file = os.path.join(workload_dir, "collected_data.json")
                best_config_file = os.path.join(workload_dir, "best_config.json")
                
                # Check for missing files
                if not os.path.exists(collected_data_file) or not os.path.exists(best_config_file):
                    continue
                
                try:
                    # Extract trailing numbers from directory name (e.g., "job_0" -> 0, "sample_twitter_config1" -> 1)
                    import re
                    match = re.search(r'_?(\d+)$', workload)
                    if not match:
                        continue
                    workload_idx = int(match.group(1))
                    
                    with open(collected_data_file, 'r') as f:
                        collected_data = json.load(f)
                    
                    with open(best_config_file, 'r') as f:
                        best_config_tmp = json.load(f)
                        
                    best_config = best_config_tmp
                    if "config" in best_config:
                        best_config = best_config["config"]

                    internal_metrics = collected_data.get('internal_metrics', {})
                    unique_query_plans = list(dict.fromkeys(collected_data.get('query_plans', [])))
                    workload_features = collected_data.get('workload_features', {})
                    
                    record = {
                        "database": db_name,
                        "hardware_specs": hw,
                        "benchmark": bench,
                        "workload_idx": workload_idx,
                        "internal_metrics": internal_metrics,
                        "query_plans": unique_query_plans,
                        "workload_features": workload_features,
                        "best_config": best_config
                    }
                    combined_data.append(record)
                    
                except Exception as e:
                    print(f"Error processing {workload_dir}: {e}")
                    
    return combined_data

In [2]:
# Combine PostgreSQL data
postgres_data = process_database('/home/E2ETune-AI4DB/data/postgresql', 'postgresql')
with open('/home/E2ETune-AI4DB/llm_tuning/postgres_combined_data.json', 'w') as f:
    json.dump(postgres_data, f, indent=2)
print(f"Successfully created postgres_combined_data.json with {len(postgres_data)} records.")

# Combine MySQL data
mysql_data = process_database('/home/E2ETune-AI4DB/data/mysql', 'mysql')
with open('/home/E2ETune-AI4DB/llm_tuning/mysql_combined_data.json', 'w') as f:
    json.dump(mysql_data, f, indent=2)
print(f"Successfully created mysql_combined_data.json with {len(mysql_data)} records.")


Successfully created postgres_combined_data.json with 4550 records.
Successfully created mysql_combined_data.json with 922 records.


In [3]:
from collections import defaultdict, Counter

postgres_counts = defaultdict(Counter)
for record in postgres_data:
    postgres_counts[record['hardware_specs']][record['benchmark']] += 1

print("PostgreSQL Counts by Hardware and Benchmark:")
for hw, bench_counts in postgres_counts.items():
    print(f"  {hw}:")
    for bench, count in bench_counts.items():
        print(f"    {bench}: {count}")

mysql_counts = defaultdict(Counter)
for record in mysql_data:
    mysql_counts[record['hardware_specs']][record['benchmark']] += 1

print("\nMySQL Counts by Hardware and Benchmark:")
for hw, bench_counts in mysql_counts.items():
    print(f"  {hw}:")
    for bench, count in bench_counts.items():
        print(f"    {bench}: {count}")

PostgreSQL Counts by Hardware and Benchmark:
  hetzner-4c-8t-32gb:
    tpch: 272
    job: 244
    ycsb: 200
    ssb: 240
    tpcds: 233
    twitter: 200
    ssb_flat_tiny: 286
    tpcc: 200
    smallbank: 200
    wikipedia: 200
  hetzner-4c-8t-64gb:
    tpch: 272
    job: 244
    ycsb: 200
    ssb: 240
    tpcds: 233
    twitter: 200
    ssb_flat_tiny: 286
    tpcc: 200
    smallbank: 200
    wikipedia: 200

MySQL Counts by Hardware and Benchmark:
  hetzner-4c-8t-32gb:
    job: 12
  hetzner-4c-8t-64gb:
    tpch: 117
    job: 34
    ssb: 240
    tpcds: 233
    ssb_flat_tiny: 286
